## Análisis exploratorio de datos (EDA) para realizar la limpieza de datos


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import SparkSession
#quiero definir para leer el csv
from pyspark.sql import DataFrameReader
from pyspark.sql.functions import col, count, isnan, when, mean, stddev, min, max, desc
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType 

In [3]:
# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("ProyectoFinalOlga-Adriana-Paula") \
    .getOrCreate()

sc = spark.sparkContext

DATA_PATH = "/home/jovyan/work/data/"

In [4]:
df_behaviour = spark.read.parquet(DATA_PATH + 'behavioural_raw_parquet')
df_clients = spark.read.parquet(DATA_PATH + 'clients_raw_parquet',)

In [8]:


# Mostrar esquema y primeras filas
print("=== CLIENTES ===")
print(f"Filas: {df_clients.count()}, Columnas: {len(df_clients.columns)}")
df_clients.printSchema()
df_clients.show(5, truncate=False)

print("\n=== COMPORTAMIENTO ===")
print(f"Filas: {df_behaviour.count()}, Columnas: {len(df_behaviour.columns)}")
df_behaviour.printSchema()
df_behaviour.show(10, truncate=False)

=== CLIENTES ===
Filas: 162977, Columnas: 45
root
 |-- CLIENT_ID: string (nullable = true)
 |-- NON_COMPLIANT_CONTRACT: integer (nullable = true)
 |-- NAME_PRODUCT_TYPE: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- TOTAL_INCOME: double (nullable = true)
 |-- AMOUNT_PRODUCT: double (nullable = true)
 |-- INSTALLMENT: double (nullable = true)
 |-- EDUCATION: string (nullable = true)
 |-- MARITAL_STATUS: string (nullable = true)
 |-- HOME_SITUATION: string (nullable = true)
 |-- REGION_SCORE: double (nullable = true)
 |-- AGE_IN_YEARS: double (nullable = true)
 |-- JOB_SENIORITY: double (nullable = true)
 |-- HOME_SENIORITY: double (nullable = true)
 |-- LAST_UPDATE: double (nullable = true)
 |-- OWN_INSURANCE_CAR: string (nullable = true)
 |-- CAR_AGE: double (nullable = true)
 |-- FAMILY_SIZE: double (nullable = true)
 |-- REACTIVE_SCORING: double (nullable = true)
 |-- PROACTIVE_SCORING: double (nullable = true)
 |-- BEHAVIORAL_SCORING: double (nullable = true)
 

In [6]:
from pyspark.sql.functions import col, count, when, isnan
from pyspark.sql.types import DoubleType, FloatType, StringType

# Función mejorada de análisis de calidad
def analyze_data_quality(df, df_name):
    print(f"\n=== CALIDAD DE DATOS: {df_name} ===")

    # Total de filas
    total_rows = df.count()

    # 1. Nulos por columna
    print("\n1. Valores nulos por columna:")
    null_counts_exprs = []

    for c in df.columns:
        dtype = df.schema[c].dataType
        if isinstance(dtype, (DoubleType, FloatType)):
            # Contar nulls + NaN
            expr = count(when(col(c).isNull() | isnan(c), c)).alias(c)
        else:
            expr = count(when(col(c).isNull(), c)).alias(c)
        
        null_counts_exprs.append(expr)

    null_counts_df = df.select(null_counts_exprs)
    null_counts_df.show(vertical=True)

    # 2. Porcentajes de nulos
    print("\n2. Porcentajes de nulos:")
    null_percentages = {}

    for c in df.columns:
        if isinstance(df.schema[c].dataType, (DoubleType, FloatType)):
            null_count = df.filter(col(c).isNull() | isnan(c)).count()
        else:
            null_count = df.filter(col(c).isNull()).count()

        percentage = (null_count / total_rows) * 100
        null_percentages[c] = percentage

        if percentage > 0:
            print(f"   {c}: {null_count} nulos ({percentage:.2f}%)")

    # 3. Filas duplicadas
    duplicate_count = df.count() - df.dropDuplicates().count()
    print(f"\n3. Filas duplicadas: {duplicate_count}")

    # 4. Posibles problemas de tipo en columnas string
    print("\n4. Columnas string que parecen numéricas:")
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            sample = (
                df.select(col(field.name))
                  .filter(col(field.name).isNotNull())
                  .limit(20)
                  .toPandas()[field.name]
                  .astype(str)
            )

            numeric_like = sample.str.match(r'^\d+(\.\d+)?$').any()

            if numeric_like:
                print(f"   {field.name}: String pero contiene valores numéricos")

    return null_percentages

# Ejecutar análisis
null_clients = analyze_data_quality(df_clients, "CLIENTES")
null_behaviour = analyze_data_quality(df_behaviour, "COMPORTAMIENTO")



=== CALIDAD DE DATOS: CLIENTES ===

1. Valores nulos por columna:
-RECORD 0-----------------------------
 CLIENT_ID                   | 0      
 NON_COMPLIANT_CONTRACT      | 0      
 NAME_PRODUCT_TYPE           | 0      
 GENDER                      | 0      
 TOTAL_INCOME                | 0      
 AMOUNT_PRODUCT              | 0      
 INSTALLMENT                 | 7      
 EDUCATION                   | 39640  
 MARITAL_STATUS              | 2      
 HOME_SITUATION              | 0      
 REGION_SCORE                | 0      
 AGE_IN_YEARS                | 0      
 JOB_SENIORITY               | 29174  
 HOME_SENIORITY              | 0      
 LAST_UPDATE                 | 0      
 OWN_INSURANCE_CAR           | 0      
 CAR_AGE                     | 107550 
 FAMILY_SIZE                 | 2      
 REACTIVE_SCORING            | 91901  
 PROACTIVE_SCORING           | 337    
 BEHAVIORAL_SCORING          | 32246  
 DAYS_LAST_INFO_CHANGE       | 1      
 NUMBER_OF_PRODUCTS          | 21903

In [7]:
# Celda 6: Estadísticas para columnas numéricas
print("=== ESTADÍSTICAS DESCRIPTIVAS - CLIENTES ===")

# Seleccionar columnas numéricas
numeric_cols = [f.name for f in df_clients.schema.fields 
                if isinstance(f.dataType, (IntegerType, DoubleType))]

# Calcular estadísticas
for col_name in numeric_cols[:10]:  # Primeras 10 para no saturar
    stats = df_clients.select(
        mean(col(col_name)).alias("media"),
        stddev(col(col_name)).alias("std"),
        min(col(col_name)).alias("min"),
        max(col(col_name)).alias("max")
    ).collect()[0]
    
    print(f"\n{col_name}:")
    print(f"  Media: {stats['media']:.2f}")
    print(f"  Std: {stats['std']:.2f}")
    print(f"  Min: {stats['min']:.2f}")
    print(f"  Max: {stats['max']:.2f}")
    
    # Contar ceros o valores atípicos
    zero_count = df_clients.filter(col(col_name) == 0).count()
    if zero_count > 0:
        print(f"  Valores cero: {zero_count}")

=== ESTADÍSTICAS DESCRIPTIVAS - CLIENTES ===

NON_COMPLIANT_CONTRACT:
  Media: 0.08
  Std: 0.27
  Min: 0.00
  Max: 1.00
  Valores cero: 149741

TOTAL_INCOME:
  Media: 2029.34
  Std: 3722.50
  Min: 307.80
  Max: 1404000.00

AMOUNT_PRODUCT:
  Media: 7193.39
  Std: 4831.53
  Min: 540.00
  Max: 48486.19

INSTALLMENT:
  Media: 325.58
  Std: 173.48
  Min: 19.39
  Max: 2898.15

REGION_SCORE:
  Media: 0.02
  Std: 0.01
  Min: 0.00
  Max: 0.06

AGE_IN_YEARS:
  Media: 43.95
  Std: 11.93
  Min: 20.52
  Max: 69.08

JOB_SENIORITY:
  Media: 2398.92
  Std: 2360.50
  Min: 1.00
  Max: 17729.00

HOME_SENIORITY:
  Media: 4987.49
  Std: 3520.92
  Min: 0.00
  Max: 24044.00
  Valores cero: 43

LAST_UPDATE:
  Media: 2992.18
  Std: 1510.22
  Min: 0.00
  Max: 6874.00
  Valores cero: 7

CAR_AGE:
  Media: 11.99
  Std: 11.80
  Min: 0.00
  Max: 64.50
  Valores cero: 1136


In [19]:
# Celda 7: Análisis de columnas categóricas
print("=== ANÁLISIS CATEGÓRICAS - CLIENTES ===")

categorical_cols = [f.name for f in df_clients.schema.fields 
                    if isinstance(f.dataType, StringType)]

for col_name in categorical_cols[:8]:  # Primeras 8
    print(f"\n{col_name}:")
    
    # Conteo de categorías
    value_counts = df_clients.groupBy(col_name).count().orderBy(desc("count"))
    
    print(f"  Número de categorías únicas: {value_counts.count()}")
    
    # Mostrar top 5 categorías
    if value_counts.count() > 10:
        print("  (Mostrando top 10 categorías)")
    
    for row in value_counts.take(10):
        print(f"    {row[col_name]}: {row['count']} ({row['count']/df_clients.count()*100:.1f}%)")

=== ANÁLISIS CATEGÓRICAS - CLIENTES ===

CLIENT_ID:
  Número de categorías únicas: 162977
  (Mostrando top 10 categorías)
    ES182234856M: 1 (0.0%)
    ES182161992N: 1 (0.0%)
    ES182369296D: 1 (0.0%)
    ES182101769P: 1 (0.0%)
    ES182300407P: 1 (0.0%)
    ES182364023V: 1 (0.0%)
    ES182258667P: 1 (0.0%)
    ES182145473H: 1 (0.0%)
    ES182330438C: 1 (0.0%)
    ES182189909E: 1 (0.0%)

NAME_PRODUCT_TYPE:
  Número de categorías únicas: 2
    PRODUCT 1: 147470 (90.5%)
    PRODUCT 2: 15507 (9.5%)

GENDER:
  Número de categorías únicas: 2
    F: 107358 (65.9%)
    M: 55619 (34.1%)

EDUCATION:
  Número de categorías únicas: 5
    Secondary: 115824 (71.1%)
    None: 39640 (24.3%)
    Incomplete University: 5407 (3.3%)
    Primary School: 2017 (1.2%)
    Master/PhD: 89 (0.1%)

MARITAL_STATUS:
  Número de categorías únicas: 3
    Married: 120022 (73.6%)
    Single: 42953 (26.4%)
    None: 2 (0.0%)

HOME_SITUATION:
  Número de categorías únicas: 6
    House: 144579 (88.7%)
    Living with r